# Séance 4 — ML supervisé (régression & classification)

**Decision problem:** when does prediction improve a decision, and when does it only create false confidence?

Official topic preserved: supervised learning. The output is a feature table and model metrics.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
SNAPSHOT = "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "snapshots" / SNAPSHOT).exists():
        ROOT = candidate
        break
OUT = ROOT / "outputs"
for d in [OUT / "bloc1", OUT / "bloc2", OUT / "bloc3", OUT / "final_product"]:
    d.mkdir(parents=True, exist_ok=True)
DATA = ROOT / "data" / "snapshots"

def load_raw_trends():
    p = DATA / "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
    df = pd.read_csv(p)
    df["date"] = pd.to_datetime(df["date"])
    for c in ["chatgpt", "iphone", "meteo"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.sort_values("date").reset_index(drop=True)

def load_clean_long():
    p = OUT / "bloc1" / "clean_trends_long.csv"
    if p.exists():
        df = pd.read_csv(p, parse_dates=["date"])
    else:
        raw = load_raw_trends()
        df = raw.melt(id_vars="date", var_name="signal", value_name="interest")
    return df.sort_values(["date", "signal"]).reset_index(drop=True)

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

In [ ]:
wide = pd.read_csv(OUT / "bloc1" / "clean_trends_wide.csv", parse_dates=["date"]) if (OUT / "bloc1" / "clean_trends_wide.csv").exists() else load_raw_trends()
for c in ["chatgpt", "iphone", "meteo"]:
    wide[f"{c}_lag1"] = wide[c].shift(1)
    wide[f"{c}_roll4"] = wide[c].rolling(4).mean()
wide["month"] = wide["date"].dt.month
threshold = wide["chatgpt"].median()
wide["target_chatgpt_high_next_week"] = (wide["chatgpt"].shift(-1) > threshold).astype(int)
features = [c for c in wide.columns if c not in ["date", "target_chatgpt_high_next_week"]]
model_data = wide.dropna().iloc[:-1].reset_index(drop=True)
model_data.to_csv(OUT / "bloc1" / "feature_table.csv", index=False)
model_data.head()

In [ ]:
split = int(len(model_data)*0.8)
X_train, y_train = model_data.loc[:split-1, features], model_data.loc[:split-1, "target_chatgpt_high_next_week"]
X_valid, y_valid = model_data.loc[split:, features], model_data.loc[split:, "target_chatgpt_high_next_week"]
base = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))]).fit(X_train, y_train)
rows=[]
for name, clf in [("baseline", base), ("logistic_pipeline", model)]:
    pred = clf.predict(X_valid)
    row={"model":name,"accuracy":accuracy_score(y_valid,pred),"f1":f1_score(y_valid,pred,zero_division=0)}
    if hasattr(clf, "predict_proba") and len(set(y_valid)) > 1:
        row["roc_auc"] = roc_auc_score(y_valid, clf.predict_proba(X_valid)[:,1])
    rows.append(row)
metrics = pd.DataFrame(rows).round(3)
metrics.to_csv(OUT / "bloc1" / "model_metrics.csv", index=False)
metrics

## Practical exercise

Identify which error type would be most costly for a product team.

## Conclusion

A model is only useful if it improves a decision against a baseline and exposes its limits.